# 🕵️ Case file 04 — The booking lookup that keeps failing

> **Mission briefing.** Last call of the day. A customer gave a valid booking reference, but the lookup tool failed. The assistant reports the failure instead of retrying. Your job is to open the tool step in the activity log, compare what the assistant sent with what the tool tried to read, and fix the tool — not the assistant's wording.

The assistant uses a tool (a booking lookup) to answer. The first run is intentionally broken; inspect its activity log before correcting it.

## Case objective
Find a failed lookup in the activity log and correct the tool.

## What to expect
The first tool asks for `booking_id`, but the assistant sends `booking_ref`, so the tool fails with an error and the answer reports it. The corrected tool asks for `booking_ref` and returns the booking. The activity logs are `activity-4-tool-call-broken` and `activity-4-tool-call-fixed`.

## Bad practice
Guess the name of the detail the tool expects, and don't check the tool step when it fails.

## Good practice
Make sure the tool asks for the same detail the assistant sends, and check the tool step's input, status, and error.

## Why the good practice works
The activity log shows what the assistant sent and what the tool tried to read. The mismatch and the error are visible, so you can fix the tool instead of the assistant's wording.

## Deliverable
Both activity-log links, the evidence from the tool step, and the tool change.


In [1]:
import json
from IPython.display import Markdown, display
from experiment_utils import RUN_ID, langfuse, run_tool_loop


## 🔮 Your prediction

The tool schema advertises `booking_ref`. The broken implementation reads `booking_id`.

**What error do you expect in the tool step?** And what will the customer see in the final answer?

Write your guess below (double-click this cell) or as a comment in the next cell.

---
*Your guess:*


In [2]:
question = "What is the status of booking SKY-1042?"
broken = run_tool_loop(question, "broken")
if isinstance(broken["tool_result"], dict) and "error" in broken["tool_result"]:
    print(f"⚠️  Lookup failed: {broken['tool_result']['error']}")
else:
    print("⚠️  The lookup did not return the booking. Check the tool step.")
display(Markdown(f"**Details sent to the tool:** `{json.dumps(broken['tool_args'])}`"))
display(Markdown(f"**Tool result:** `{json.dumps(broken['tool_result'])}`"))
display(Markdown(f"**Final answer:** {broken['answer']}"))
display(Markdown(f"[Open activity log]({broken['trace_url']})"))


⚠️  Lookup failed: KeyError: 'booking_id'


**Details sent to the tool:** `{"booking_ref": "SKY-1042"}`

**Tool result:** `{"error": "KeyError: 'booking_id'"}`

**Final answer:** I tried to look up booking **SKY-1042**, but the lookup failed with this error:

> `KeyError: 'booking_id'`

This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/42665213acbba1f4f13648936f4a373d)

## 🔍 What went wrong?

Open the first activity log and look at the `get_booking` tool step. Compare the details the assistant sent with the details the tool tried to read. The schema says `booking_ref`; the implementation reads `booking_id`. Correct the tool, not the assistant's wording.


In [3]:
fixed = run_tool_loop(question, "fixed")
if isinstance(fixed["tool_result"], dict) and "error" not in fixed["tool_result"]:
    print(f"✅  Lookup succeeded: booking status is {fixed['tool_result'].get('status', 'found')}.")
else:
    print("⚠️  The lookup still did not return the booking. Check the tool step.")
display(Markdown(f"**Details sent to the tool:** `{json.dumps(fixed['tool_args'])}`"))
display(Markdown(f"**Tool result:** `{json.dumps(fixed['tool_result'])}`"))
display(Markdown(f"**Final answer:** {fixed['answer']}"))
display(Markdown(f"[Open activity log]({fixed['trace_url']})"))
rows = ["| run | details sent to the tool | tool result/error | final answer | activity log |",
        "|---|---|---|---|---|"]
for run in [broken, fixed]:
    args = json.dumps(run["tool_args"]).replace("|", "\\|")
    result = json.dumps(run["tool_result"]).replace("|", "\\|")
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {args} | {result} | {answer} | [Open log]({run['trace_url']}) |")
display(Markdown("\n".join(rows)))


✅  Lookup succeeded: booking status is confirmed.


**Details sent to the tool:** `{"booking_ref": "SKY-1042"}`

**Tool result:** `{"status": "confirmed", "origin": "Lisbon", "destination": "Madrid", "date": "2026-12-01", "passengers": 2, "cabin": "economy"}`

**Final answer:** Booking **SKY-1042** is **confirmed**.

- **Route:** Lisbon → Madrid
- **Date:** December 1, 2026
- **Passengers:** 2
- **Cabin:** Economy

Let me know if you'd like any other details about this booking.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/22b499b3ecca6f608386e5a6bb2691bd)

| run | details sent to the tool | tool result/error | final answer | activity log |
|---|---|---|---|---|
| before fix | {"booking_ref": "SKY-1042"} | {"error": "KeyError: 'booking_id'"} | I tried to look up booking **SKY-1042**, but the lookup failed with this error:  > `KeyError: 'booking_id'`  This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more. | [Open log](http://localhost:3000/project/langfuse-observability/traces/42665213acbba1f4f13648936f4a373d) |
| after fix | {"booking_ref": "SKY-1042"} | {"status": "confirmed", "origin": "Lisbon", "destination": "Madrid", "date": "2026-12-01", "passengers": 2, "cabin": "economy"} | Booking **SKY-1042** is **confirmed**.  - **Route:** Lisbon → Madrid - **Date:** December 1, 2026 - **Passengers:** 2 - **Cabin:** Economy  Let me know if you'd like any other details about this booking. | [Open log](http://localhost:3000/project/langfuse-observability/traces/22b499b3ecca6f608386e5a6bb2691bd) |

## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [4]:
lines = [f"**Session:** `activity-4-{RUN_ID}`", "", "**Activity logs:**", ""]
for run in [broken, fixed]:
    lines.append(f"- [{run['label']}]({run['trace_url']})")
display(Markdown("\n".join(lines)))
langfuse.flush()
print("✅ Incident closed: `get_booking` now reads the field the assistant actually sends.")
print("🕵️ Investigation Tasks: 4/4 completed. Case file closed — nice work, Detective.")
display(Markdown("Activity logs sent to Langfuse. Open them and compare the tool steps."))


**Session:** `activity-4-2c480758`

**Activity logs:**

- [before fix](http://localhost:3000/project/langfuse-observability/traces/42665213acbba1f4f13648936f4a373d)
- [after fix](http://localhost:3000/project/langfuse-observability/traces/22b499b3ecca6f608386e5a6bb2691bd)

✅ Incident closed: `get_booking` now reads the field the assistant actually sends.
🕵️ Investigation Tasks: 4/4 completed. Case file closed — nice work, Detective.


Activity logs sent to Langfuse. Open them and compare the tool steps.

## 🎁 Bonus — What else would you try?

- Validate tool arguments against the schema before the call, and log the rejection.
- Add a friendly fallback for a missing booking instead of an error.
- Search the trace for the tool step's `level` and `status_message`: what would an alert on those fields catch?


## 🎯 Final check — scored quiz

One question, four options, one answer. Your score combines correctness with speed:

| Result | Points |
|---|---|
| Correct answer | 60 base points |
| Speed bonus | up to 40 points, minus 1 per whole second from quiz start |
| Wrong answer | 0 points |

Run the next cell to start the timer and see the question. Then set `answer` to **1, 2, 3, or 4** in the cell after it and run that cell once.

The checked-in output shows 100 because verification ran both cells back-to-back with the default (correct) answer; your own run measures your choice and speed.

**Report the printed score for this case.** Organizers sum the four case scores (maximum 400) to rank participants.


In [5]:
import time

quiz_started = time.time()

print("🎯 Quiz — one question, one answer")
print()
print("What caused the failed booking lookup?")
print("  1. The customer gave an invalid booking reference.")
print("  2. The assistant called the wrong tool.")
print("  3. The tool read booking_id, while the assistant sent booking_ref.")
print("  4. The Langfuse trace was incomplete.")
print()
print("Timer started. Set answer in the next cell to 1, 2, 3, or 4, then run it.")


🎯 Quiz — one question, one answer

What caused the failed booking lookup?
  1. The customer gave an invalid booking reference.
  2. The assistant called the wrong tool.
  3. The tool read booking_id, while the assistant sent booking_ref.
  4. The Langfuse trace was incomplete.

Timer started. Set answer in the next cell to 1, 2, 3, or 4, then run it.


In [6]:
answer = 3  # replace with your choice: 1, 2, 3, or 4

if "quiz_started" not in globals():
    raise RuntimeError("Run the quiz start cell first.")
if answer not in (1, 2, 3, 4):
    raise ValueError("Set answer to 1, 2, 3, or 4.")

elapsed = time.time() - quiz_started
correct_option = 3
correct = answer == correct_option
base = 60 if correct else 0
speed_bonus = max(0, 40 - int(elapsed)) if correct else 0
score = base + speed_bonus

if correct:
    print(f"✅ Correct. Base {base} + speed bonus {speed_bonus} = {score}/100.")
else:
    print(f"❌ Not quite. The correct option is {correct_option}. Score: {score}/100.")
print(f"⏱️  Time from quiz start: {elapsed:.1f}s")
print(f"📋 Score to report for this case: {score} (max 100; organizers sum all four cases)")


✅ Correct. Base 60 + speed bonus 40 = 100/100.
⏱️  Time from quiz start: 0.0s
📋 Score to report for this case: 100 (max 100; organizers sum all four cases)
